# V1 / V2 模型比较与优化效果分析

本 Notebook 基于 v1 错误样本分析结果，对优化后的 v2 模型进行对比评估。报告采用与 `model_error_and_metrics_analysis.ipynb` 一致的图文结构：先给出图表，再在对应图片下方解释结果、原因和解决方案。

## 结论摘要

- v2 相比 v1 的整体指标有稳定提升：Accuracy 从 `0.9084` 提升到 `0.9237`，Macro-F1 从 `0.9176` 提升到 `0.9299`。
- v2 对 v1 的弱类有明显改善，尤其是 `trash`、`paper`、`cardboard`、`plastic`。
- v2 的提升不是大幅跃升，但对一个原本已经超过 90% accuracy 的模型而言，约 1.5 个百分点的测试集提升是有效的。
- v2 仍不建议直接覆盖 v1，应作为独立版本保存，等前后端同学确认模型路径、类别顺序和输入尺寸后再切换。

In [ ]:
from pathlib import Path
import json
import pandas as pd

base = Path('.')
comparison = pd.read_csv('supporting_files/v1_v2_overall_comparison.csv')
weak = pd.read_csv('supporting_files/v1_v2_weak_class_f1.csv')
v2_metrics = json.loads(Path('../../model/model_v2/outputs/convnext_tiny_v2/test_metrics_v2.json').read_text(encoding='utf-8'))

## 图 1：V1 与 V2 整体指标对比

![V1 vs V2 Overall Metrics](figures/v1_v2_overall_metrics.png)

**结果解读：** v2 在所有主要指标上均优于 v1。Accuracy 从 `0.9084` 提升到 `0.9237`，提升约 `1.53` 个百分点；Macro-F1 从 `0.9176` 提升到 `0.9299`，说明提升不是只来自样本较多的大类，而是各类别平均表现也有所改善。Macro ROC-AUC 从 `0.9834` 提升到 `0.9897`，代表模型对各类别的排序区分能力进一步增强。

**原因分析：** v1 已经具备较强的基础分类能力，因此 v2 的提升幅度不会特别夸张。v2 的提升主要来自三方面：第一，Focal Loss 让模型更关注难分类样本；第二，困难类别加权和 WeightedRandomSampler 提高了易错类别的训练有效权重；第三，补充样本增加了模型对相似材质和复杂背景的见识。

**评估结论：** v2 优化有效，且没有牺牲整体泛化能力。由于 validation/test 没有加入补充样本，因此测试集提升可以视为真实泛化能力提升，而不是评估集污染。

In [ ]:
comparison

## 图 2：弱类别 F1-score 对比

![Weak Class F1 Improvement](figures/v1_v2_weak_class_f1.png)

**结果解读：** v1 中表现较弱的类别在 v2 中均有提升。`trash` 的 F1 从 `0.8400` 提升到 `0.8784`，改善最明显；`paper` 从 `0.8619` 提升到 `0.8918`；`cardboard` 从 `0.8610` 提升到 `0.8908`；`plastic` 从 `0.8288` 提升到 `0.8507`。`glass` 原本已经较好，因此只小幅提升。

**原因分析：** v1 的错误主要集中在 `cardboard/paper`、`plastic/trash`、`plastic/glass` 等材质相似或语义边界模糊的组合。v2 针对这些类别加入补充样本和更强增强，使模型看到更多形态变化；同时 Focal Loss 降低容易样本对损失的主导作用，使难例对参数更新的影响更大。

**仍然存在的问题：** `plastic` 虽有提升，但 F1 仍是这几个弱类中较低的一个，说明透明塑料、塑料袋、塑料包装与 glass/trash 的边界仍不稳定。后续如果继续优化，优先补充更高质量的 plastic 难例会比盲目增加所有类别数据更有效。

In [ ]:
weak

## 图 3：V2 训练曲线

![V2 Training Curve](figures/v2_training_curve.png)

**结果解读：** v2 训练过程中 train accuracy 持续上升，validation accuracy 也整体上升，并在后期达到较好的验证表现。训练集准确率明显高于验证集，说明后期存在一定过拟合迹象；但 validation accuracy 没有崩掉，最终测试集也达到 `0.9237`，因此过拟合处于可接受范围。

**原因分析：** ConvNeXt-Tiny 表达能力较强，训练到后期很容易把训练集拟合得很高。v2 使用了较强数据增强、Focal Loss 和 early stopping 思路来缓解过拟合。补充样本只加入训练集而不加入验证/测试集，也能在不污染评估的前提下增加训练多样性。

**解决方案：** 如果之后继续训练，可以保留 early stopping；当 validation accuracy 多轮不提升时停止。若想进一步降低过拟合，可以尝试更强 weight decay、降低 epoch、减小 hard-class weight，或只保留质量更高的补充样本。

## 图 4：V2 混淆矩阵

![V2 Confusion Matrix](../../model/model_v2/outputs/convnext_tiny_v2/confusion_matrix_v2.png)

**结果解读：** v2 混淆矩阵整体对角线较明显，说明大多数样本仍被正确分类。错分主要仍集中在纸类、塑料、垃圾、玻璃、金属等视觉边界接近的类别之间，这与 v1 的错误结构一致，但错误程度有所下降。

**原因分析：** 模型对明确类别，如 `clothes`、`shoes`，表现非常稳定；而 `plastic`、`trash`、`paper` 这类类别内部变化大、边界模糊，仍容易混淆。尤其 trash 本身常包含纸、塑料、金属等混合物，类别定义天然更复杂。

**解决方案：** 对混淆矩阵中的非对角线热点进行人工抽样复核。如果发现某些 trash 样本其实更像 paper/plastic，应统一标注标准；如果标注无误，则继续补充这类边界样本，并可在推理端输出 top-k 预测辅助人工判断。

## V2 模型改进方案

v2 的改进不是单一操作，而是围绕 v1 错误分析设计的一组组合方案：

1. **保留 v1，不直接覆盖。** v2 输出到 `model/model_v2/`，避免影响其他同学现有后端、前端或部署流程。
2. **使用 Focal Loss。** v1 的高置信错误说明模型对部分难例过度自信，Focal Loss 可以让训练更关注这些难例。
3. **提高困难类别权重。** v1 的弱类集中在 `cardboard`、`paper`、`plastic`、`trash`、`glass`，所以 v2 设置 `hard_class_weight=1.8`。
4. **使用 WeightedRandomSampler。** 让困难类别和难样本在训练中被更充分采样，减少模型只学容易样本的问题。
5. **加入补充样本。** 额外加入 `583` 张补充样本，覆盖 cardboard、glass、metal、paper、plastic、trash。补充样本只进入训练集，不进入 validation/test。
6. **加强数据增强。** 使用更强随机裁剪、旋转、颜色扰动、透视变化、Random Erasing，提升模型对真实场景变化的鲁棒性。
7. **保持评估集干净。** `split_data/val` 和 `split_data/test` 不做修改，保证 v1/v2 对比公平。

**方案有效性：** 从结果看，v2 的 Accuracy、Macro-F1、Weighted-F1、ROC-AUC 均提升，弱类 F1 也普遍提升，因此该方案是有效的。

## 后续优化建议

- 继续补充 `plastic` 的高质量难例，重点是透明塑料、塑料袋、瓶身反光、复杂背景包装。
- 对 `trash` 类别进行标注复核，统一哪些混合垃圾算 trash、哪些应归入 paper/plastic/metal。
- 若继续训练 v3，可尝试更小的 hard-class weight，例如 `1.4` 或 `1.6`，比较是否能进一步提升 plastic recall。
- 部署时保留 v1 和 v2 两套权重，先让后端通过配置切换，不要直接替换旧文件。
- GitHub 不建议上传 `.pt` 权重文件；权重可以放 Kaggle output、网盘或 Git LFS。